In [2]:
!pip install --upgrade --quiet json-repair networkx  geopandas folium contextily


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install -U "langchain==0.2.8" "langchain-community==0.2.7" "langchain-core==0.2.19" "langchain-experimental==0.0.62" "langchain-openai "

  Using cached numpy-1.26.4-cp39-cp39-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to r

In [6]:
!pip install -U neo4j langchain-neo4j

  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl.metadata (60 kB)
  Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (60 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.8 MB/s eta 0:00:00
Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl (30.3 MB)
Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.6/640.6 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 8.5.0
    Uninstalling tenacity-8.5.0:
      Successfully uninstalled tenacity-8.5.0
  Attempting uninstall: numpym━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [pypdf]
    Found existing installation: numpy 1.26.4━━━━━━━━━━━━━━━━━  3/14 [pypdf]
    Uninstalling numpy-1.26.4:━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [numpy]
      Successfully uninstalled numpy-1.26.4━━━━━━━━

In [16]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

In [108]:
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="test1234",
)

In [18]:
graph.query("RETURN 1 AS ok")

[{'ok': 1}]

In [19]:
import json
import pandas as pd 
import numpy as np
pd.options.display.max_columns = None


In [20]:
with open('../DB/resultados.json', 'r') as f:
  data = json.load(f)

In [21]:
columns = ['place_id',
           'title',
           'gps_coordinates',
           'rating',
           'reviews',
           'price',
           'type',
           'address',
           'operating_hours',
           'phone',
           'user_review',
           'website',
           'highlights',
           'popular_for',
           'offerings',
           'amenities',
           'atmosphere',
           'crowd',
           'planning',
           'payments',
           'children',
           'service_options']
series = []
for i in range(len(data['local_results'])):
  row_data = {col: None for col in columns}
  for col in columns:
    try:
      row_data[col] = data['local_results'][i][col]
      if col == 'gps_coordinates' and row_data[col] != None:
        row_data['latitude'] = row_data[col]['latitude']
        row_data['longitude'] = row_data[col]['longitude']
    except:
      pass
  series.append(pd.Series(row_data))

In [22]:
df = pd.DataFrame(series)

In [23]:
df.head(1)

,place_id,title,gps_coordinates,rating,reviews,price,type,address,operating_hours,phone,user_review,website,highlights,popular_for,offerings,amenities,atmosphere,crowd,planning,payments,children,service_options,latitude,longitude
0,ChIJl4pAOQBb04UR6K2PbfKH85s,VegCo,"{'latitude': 20.597718999999998, 'longitude': ...",4.6,378,MX$200–300,Vegan restaurant,"J. Asunción Romero 1, San Javier, 76020 Santia...","{'tuesday': '9:30 AM–9 PM', 'wednesday': '9:30...",+52 442 434 1180,"""Delicious vegan restaurant with the best serv...",http://www.facebook.com/vegco.mx,None,None,None,None,None,None,None,None,None,"{'dine_in': True, 'takeout': True, 'delivery':...",20.597719,-100.380931


In [137]:
# -*- coding: utf-8 -*-
"""
Carga de restaurantes a Neo4j con Neo4jGraph (LangChain):
- Borra todo antes de cargar.
- Crea constraints/índices.
- Inserta :Restaurant con TODAS las propiedades (incluye user_review).
- Crea relaciones de catálogo (Amenity, ServiceOption, Offering, etc.).
- Conecta a :City {name:"Santiago de Querétaro"} con aliases.
- Guarda lat/lon y point en Restaurant y calcula centroide en City.
- *** Unifica tipos de restaurantes a categorías canónicas (ES/EN) ***
"""
import math
import re
import ast


# ---------------------------
# Configuración de ciudad
# ---------------------------
CITY_NAME = "Santiago de Querétaro"
CITY_ALIASES = [
    "Santiago de Querétaro",
    "Santiago de Queretaro",
    "Querétaro",
    "Queretaro",
    "Qro",
    "Qro.",
    "Queretarock",
]

# ---------------------------
# Reglas de categorías canónicas (ES/EN)
# Cada tupla: (regex, "EtiquetaCanonical")
# NOTA: Son "OR" semánticos; puedes ampliar/ajustar libremente.
# ---------------------------
CANON_CATEGORY_RULES = [
    (r"\bvegan|vegano|vegana",                           "Vegan"),
    (r"\bvegetar",                                       "Vegetarian"),
    (r"\bgluten[- ]?free|sin gluten",                    "Gluten-free"),
    (r"\bhealthy|saludabl|health\s*food",               "Healthy"),
    (r"\bbrunch",                                        "Brunch"),
    (r"\basian|asiátic",                                 "Asian"),
    (r"\bsushi",                                         "Sushi"),
    (r"\bthai|tailand",                                  "Thai"),
    (r"\bjapan|japon|japonés|japones",                   "Japanese"),
    (r"\bchinese|chin(a|o)|china",                       "Chinese"),
    (r"\bkorean|corean",                                 "Korean"),
    (r"\bindian|indio|hind[uú]",                         "Indian"),
    (r"\bmexican|mexican[a|o]|taquer[ií]a|taco",        "Mexican"),
    (r"\bitalian|italian[a|o]|trattoria",                "Italian"),
    (r"\bpizz",                                          "Pizza"),
    (r"\bpasta",                                         "Pasta"),
    (r"\bburger|hamburgues",                             "Burger"),
    (r"\bsteak|parrill|asador|steakhouse",               "Steakhouse"),
    (r"\bbar\s*&?\s*grill|grill|brasa",                  "Bar & Grill"),
    (r"\bsea\s*food|marisc|cevicher",                    "Seafood"),
    (r"\bbbq|barbacoa|ahumad|smokehouse",                "BBQ"),
    (r"\bmediterranean|mediterr[aá]ne",                  "Mediterranean"),
    (r"\bspanish|español|tapas",                         "Spanish"),
    (r"\bfrench|franc[eé]s|bistr[oó]",                   "French"),
    (r"\bleban|middle\s*east|arab|árabe|turk",           "Middle Eastern"),
    (r"\bperu|peruano|cevich",                           "Peruvian"),
    (r"\bbrazil|brasil|rodizio|churrasc",                "Brazilian"),
    (r"\bargent|argentino|asado",                        "Argentinian"),
    (r"\bcafe|cafeter[ií]a|coffee",                      "Cafe"),
    (r"\bbakery|panader|pasteler",                       "Bakery"),
    (r"\bdessert|postre|reposter",                       "Dessert"),
    (r"\bice\s*cream|helader",                           "Ice Cream"),
    (r"\btea|t[eé]ter[ií]a",                             "Tea House"),
    (r"\bpub|cantina|taberna",                           "Pub"),
    (r"\bbar(?!\s*&\s*grill)|coctel|cocktail",           "Bar"),
    (r"\bwine\s*bar|vinotec",                            "Wine Bar"),
    (r"\bbrew|cervecer|taproom",                         "Brewery"),
    (r"\bgastropub",                                     "Gastropub"),
    (r"\bbuffet",                                        "Buffet"),
    (r"\bfamily|familiar",                               "Family Style"),
    (r"\bfast\s*food|comida\s*r[aá]pida",                "Fast Food"),
    (r"\bfine\s*dining|alta\s*cocina",                   "Fine Dining"),
    (r"\bfood\s*court|patio\s*de\s*comidas",             "Food Court"),
    (r"\btake\s*away|para\s*llevar|takeout",             "Takeaway"),
    (r"\bdeli|charcuter",                                "Deli"),
    (r"\bbreakfast|desayuno",                            "Breakfast"),
    (r"\blunch|comida\b",                                "Lunch"),
    (r"\bdinner|cena",                                   "Dinner"),
]

def canonicalize_categories(raw_categories, title=None):
    """Genera lista deduplicada de categorías canónicas a partir de:
       - categorías originales (lista)
       - título opcional (para inferencias por nombre)
    """
    text = " | ".join([str(x).lower() for x in (raw_categories or []) if isinstance(x, str)])
    if title:
        text += " | " + str(title).lower()
    found = set()
    for rx, label in CANON_CATEGORY_RULES:
        if re.search(rx, text, flags=re.IGNORECASE):
            found.add(label)
    # Si no detecta nada pero hay alguna mención fuerte genérica:
    if not found and re.search(r"\brestaurant|restaurante", text, flags=re.IGNORECASE):
        # No canonizamos "Restaurant" para evitar ruido; lo dejamos vacío.
        pass
    return sorted(found)

# ---------------------------
# Utilidades de limpieza
# ---------------------------
def is_nan(x):
    return isinstance(x, float) and math.isnan(x)

def to_float(x):
    try:
        if x is None or is_nan(x):
            return None
        return float(str(x).replace("★", "").strip())
    except Exception:
        return None

def to_int(x):
    try:
        if x is None or is_nan(x):
            return None
        return int(float(x))
    except Exception:
        return None

_SPLIT_RX = re.compile(r"[|,;/]")

def to_list(v):
    """Convierte a lista normalizada; acepta dict/list/tuple/set/str."""
    if v is None or is_nan(v):
        return []
    if isinstance(v, (list, tuple, set)):
        return [str(x).strip() for x in v if str(x).strip()]
    if isinstance(v, dict):
        # Representación "k:v" para no perder información
        return [f"{k}:{v}" for k, v in v.items()]
    s = str(v).strip()
    if not s:
        return []
    parts = [p.strip() for p in _SPLIT_RX.split(s) if p.strip()]
    return parts

def service_options_to_list(v):
    """Si viene dict con booleans, solo las opciones True."""
    if isinstance(v, dict):
        return [k for k, ok in v.items() if ok]
    return to_list(v)

def to_reviews_list(v):
    """
    Soporta 0..n reseñas en una misma columna:
    - dict/list/tuple/set -> lista de strings
    - string con separadores -> lista
    - string simple -> [string]
    """
    if v is None or is_nan(v):
        return []
    if isinstance(v, (list, tuple, set)):
        return [str(x).strip() for x in v if str(x).strip()]
    if isinstance(v, dict):
        # Convierte dict a JSON legible
        return [json.dumps(v, ensure_ascii=False)]
    s = str(v).strip()
    if not s:
        return []
    # Permite dividir múltiples reseñas en una celda
    candidates = [p.strip() for p in _SPLIT_RX.split(s) if p.strip()]
    return candidates if len(candidates) > 1 else [s]

def parse_hours(v):
    """Devuelve dict con llaves (monday..sunday) si es posible; si no, {}."""
    if isinstance(v, dict):
        return v
    if isinstance(v, str) and v.strip():
        # Intento literal_eval luego JSON
        for loader in (ast.literal_eval, json.loads):
            try:
                obj = loader(v)
                return obj if isinstance(obj, dict) else {}
            except Exception:
                pass
    return {}

def rating_band_label(r):
    """Banda de rating en saltos de 0.5 (e.g., 4.5–5.0)."""
    if r is None:
        return None
    try:
        r = float(r)
    except Exception:
        return None
    lo = max(0.0, round(math.floor(r * 2) / 2, 1))
    hi = min(5.0, round(lo + 0.5, 1))
    return f"{lo:.1f}–{hi:.1f}"

def canonical_city(_address: str) -> str:
    """Todos los nodos van a la ciudad canónica (según requerimiento)."""
    return CITY_NAME

# ---------------------------
# Infra: ejecución segura
# ---------------------------
def run(cypher: str, params: dict = None, silent: bool = False):
    try:
        return graph.query(cypher, params or {})
    except Exception as e:
        if not silent:
            print(f"[WARN] Cypher falló: {e}\n---\n{cypher}\n---")
        return []

# ---------------------------
# Reset + constraints/índices + ciudad
# ---------------------------
def reset_database():
    # 1) Borrar todo
    run("MATCH (n) DETACH DELETE n")

    # 2) Constraints/Índices
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.place_id IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:City)       REQUIRE c.name IS UNIQUE")

    # Catálogos
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Category)       REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:PriceRange)     REQUIRE x.label IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:RatingBand)     REQUIRE x.band  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Amenity)        REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:ServiceOption)  REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Offering)       REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Highlight)      REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:PopularFor)     REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Atmosphere)     REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Crowd)          REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Planning)       REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Payment)        REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:ChildrenOption) REQUIRE x.name  IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Website)        REQUIRE x.url   IS UNIQUE")
    run("CREATE CONSTRAINT IF NOT EXISTS FOR (x:Phone)          REQUIRE x.number IS UNIQUE")

    # (Opcional) Constraint compuesto para Review (Neo4j 5):
    run("""
    CREATE CONSTRAINT IF NOT EXISTS
    FOR (rev:Review) REQUIRE (rev.place_id, rev.text) IS NODE KEY
    """, silent=True)  # Silencio por compatibilidad si tu versión no soporta NODE KEY

    # Índices
    run("CREATE RANGE INDEX IF NOT EXISTS FOR (r:Restaurant) ON (r.rating)")
    run("CREATE POINT INDEX IF NOT EXISTS FOR (r:Restaurant) ON (r.location)")
    run("CREATE POINT INDEX IF NOT EXISTS FOR (c:City)       ON (c.location)")

    # 3) Ciudad con aliases
    run("""
        MERGE (c:City {name: $name})
        ON CREATE SET c.aliases = $aliases, c.createdAt = timestamp()
        ON MATCH  SET c.aliases = $aliases
    """, {"name": CITY_NAME, "aliases": CITY_ALIASES})

# ---------------------------
# Preparación de filas desde df
# ---------------------------
def build_rows_from_df(df: pd.DataFrame):
    rows = []
    for _, r in df.iterrows():
        title = str(r.get("title") or "").strip()
        if not title:
            continue

        addr = r.get("address")

        # Lat/Lon (o desde gps_coordinates)
        lat = to_float(r.get("latitude"))
        lon = to_float(r.get("longitude"))
        gps_raw = r.get("gps_coordinates")
        gps_json = None
        if (lat is None or lon is None) and isinstance(gps_raw, (dict, str)):
            gc = gps_raw
            try:
                if isinstance(gc, str):
                    gc = ast.literal_eval(gc)
                if isinstance(gc, dict):
                    lat = lat if lat is not None else to_float(gc.get("latitude"))
                    lon = lon if lon is not None else to_float(gc.get("longitude"))
            except Exception:
                pass
        # preserva gps_coordinates como JSON string si existe
        if isinstance(gps_raw, (dict, list)):
            gps_json = json.dumps(gps_raw, ensure_ascii=False)
        elif isinstance(gps_raw, str) and gps_raw.strip():
            gps_json = gps_raw.strip()

        # Campos complejos
        hours        = parse_hours(r.get("operating_hours"))
        highlights   = to_list(r.get("highlights"))
        popular_for  = to_list(r.get("popular_for"))
        offerings    = to_list(r.get("offerings"))
        amenities    = to_list(r.get("amenities"))
        atmosphere   = to_list(r.get("atmosphere"))
        crowd        = to_list(r.get("crowd"))
        planning     = to_list(r.get("planning"))
        payments     = to_list(r.get("payments"))
        children     = to_list(r.get("children"))
        service_opts = service_options_to_list(r.get("service_options"))
        categories   = to_list(r.get("type"))  # puede contener p.ej. "Vegan restaurant"

        # *** NUEVO: categorías canónicas ***
        canonical_categories = canonicalize_categories(categories, title=title)

        price_label = None if pd.isna(r.get("price")) else str(r.get("price")).strip()
        rating_val  = to_float(r.get("rating"))
        rating_band = rating_band_label(rating_val)
        reviews_cnt = to_int(r.get("reviews"))

        # Soporta 0..n reviews en user_review
        user_reviews = to_reviews_list(r.get("user_review"))

        rows.append({
            "title": title,
            "place_id": (r.get("place_id") or title),  # fallback si faltara
            "rating": rating_val,
            "rating_band": rating_band,
            "reviews": reviews_cnt,
            "price": price_label,
            "categories": categories,
            "canonical_categories": canonical_categories,  # <-- NUEVO

            "type": None if pd.isna(r.get("type")) else str(r.get("type")),
            "address": None if pd.isna(addr) else str(addr),
            "phone": None if pd.isna(r.get("phone")) else str(r.get("phone")),
            "website": None if pd.isna(r.get("website")) else str(r.get("website")),

            # Guarda la primera review como propiedad; todas irán como nodos
            "user_review_prop": user_reviews[0] if user_reviews else None,
            "user_reviews": user_reviews,  # lista

            "lat": lat,
            "lon": lon,
            "gps_coordinates": gps_json,
            "hours": hours,

            "highlights": highlights,
            "popular_for": popular_for,
            "offerings": offerings,
            "amenities": amenities,
            "atmosphere": atmosphere,
            "crowd": crowd,
            "planning": planning,
            "payments": payments,
            "children": children,
            "service_options": service_opts,

            "city": canonical_city(addr),
        })
    return rows

# ---------------------------
# Ingesta principal
# ---------------------------
def ingest_dataframe(df: pd.DataFrame):
    reset_database()
    rows = build_rows_from_df(df)

    cypher = """
    UNWIND $rows AS row

    // --- Nodo principal ---
    MERGE (r:Restaurant {place_id: row.place_id})
    SET r.title        = row.title,
        r.rating       = row.rating,
        r.reviews      = row.reviews,
        r.price        = row.price,
        r.type         = row.type,
        r.address      = row.address,
        r.phone        = row.phone,
        r.website      = row.website,
        r.user_review  = row.user_review_prop,        // propiedad resumida
        r.gps_raw      = row.gps_coordinates,         // preserva input
        r.latitude     = CASE WHEN row.lat IS NOT NULL THEN toFloat(row.lat) ELSE r.latitude END,
        r.longitude    = CASE WHEN row.lon IS NOT NULL THEN toFloat(row.lon) ELSE r.longitude END,
        r.location     = CASE
                           WHEN row.lat IS NOT NULL AND row.lon IS NOT NULL
                           THEN point({latitude: row.lat, longitude: row.lon})
                           ELSE r.location
                         END,
        // Horarios por día (si existen)
        r.hours_monday    = coalesce(row.hours.monday,    r.hours_monday),
        r.hours_tuesday   = coalesce(row.hours.tuesday,   r.hours_tuesday),
        r.hours_wednesday = coalesce(row.hours.wednesday, r.hours_wednesday),
        r.hours_thursday  = coalesce(row.hours.thursday,  r.hours_thursday),
        r.hours_friday    = coalesce(row.hours.friday,    r.hours_friday),
        r.hours_saturday  = coalesce(row.hours.saturday,  r.hours_saturday),
        r.hours_sunday    = coalesce(row.hours.sunday,    r.hours_sunday)

    // --- Ciudad canónica + relación ---
    WITH r, row
    MERGE (c:City {name: $city_name})
      ON CREATE SET c.aliases = $city_aliases, c.createdAt = timestamp()
      ON MATCH  SET c.aliases = $city_aliases
    MERGE (r)-[:LOCATED_IN]->(c)

    // --- Category / type (ORIGINALES) ---
    WITH r, row
    UNWIND coalesce(row.categories, []) AS cat
    MERGE (t:Category {name: cat})
      ON CREATE SET t.canonical = false
      ON MATCH  SET t.canonical = coalesce(t.canonical, false)
    MERGE (r)-[r_ot:OF_TYPE]->(t)
      ON CREATE SET r_ot.canonical = false
      ON MATCH  SET r_ot.canonical = coalesce(r_ot.canonical, false)

    // --- Category / type (CANÓNICAS) ---
    WITH r, row
    UNWIND coalesce(row.canonical_categories, []) AS canon
    MERGE (tc:Category {name: canon})
      ON CREATE SET tc.canonical = true
      ON MATCH  SET tc.canonical = true
    MERGE (r)-[r_tc:OF_TYPE]->(tc)
      ON CREATE SET r_tc.canonical = true
      ON MATCH  SET r_tc.canonical = true

    // --- Price range ---
    WITH r, row
    FOREACH (price IN CASE WHEN row.price IS NULL THEN [] ELSE [row.price] END |
        MERGE (pr:PriceRange {label: price})
        MERGE (r)-[:IN_PRICE_RANGE]->(pr)
    )

    // --- Rating band ---
    WITH r, row
    FOREACH (band IN CASE WHEN row.rating_band IS NULL THEN [] ELSE [row.rating_band] END |
        MERGE (rb:RatingBand {band: band})
        MERGE (r)-[:IN_RATING_BAND]->(rb)
    )

    // --- Website / Phone ---
    WITH r, row
    FOREACH (url IN CASE WHEN row.website IS NULL THEN [] ELSE [row.website] END |
        MERGE (w:Website {url: url})
        MERGE (r)-[:HAS_WEBSITE]->(w)
    )
    WITH r, row
    FOREACH (ph IN CASE WHEN row.phone IS NULL THEN [] ELSE [row.phone] END |
        MERGE (p:Phone {number: ph})
        MERGE (r)-[:HAS_PHONE]->(p)
    )

    // --- Amenidades ---
    WITH r, row
    UNWIND coalesce(row.amenities, []) AS am
    MERGE (a:Amenity {name: am})
    MERGE (r)-[:HAS_AMENITY]->(a)

    // --- Opciones de servicio ---
    WITH r, row
    UNWIND coalesce(row.service_options, []) AS so
    MERGE (s:ServiceOption {name: so})
    MERGE (r)-[:OFFERS_SERVICE]->(s)

    // --- Ofertas / offerings ---
    WITH r, row
    UNWIND coalesce(row.offerings, []) AS off
    MERGE (o:Offering {name: off})
    MERGE (r)-[:HAS_OFFERING]->(o)

    // --- Highlights ---
    WITH r, row
    UNWIND coalesce(row.highlights, []) AS hi
    MERGE (h:Highlight {name: hi})
    MERGE (r)-[:HAS_HIGHLIGHT]->(h)

    // --- Popular for ---
    WITH r, row
    UNWIND coalesce(row.popular_for, []) AS pf
    MERGE (p:PopularFor {name: pf})
    MERGE (r)-[:IS_POPULAR_FOR]->(p)

    // --- Atmósfera ---
    WITH r, row
    UNWIND coalesce(row.atmosphere, []) AS atm
    MERGE (a2:Atmosphere {name: atm})
    MERGE (r)-[:HAS_ATMOSPHERE]->(a2)

    // --- Tipo de clientela (crowd) ---
    WITH r, row
    UNWIND coalesce(row.crowd, []) AS cr
    MERGE (c2:Crowd {name: cr})
    MERGE (r)-[:ATTRACTS_CROWD]->(c2)

    // --- Planeación (reservas, etc.) ---
    WITH r, row
    UNWIND coalesce(row.planning, []) AS pl
    MERGE (p2:Planning {name: pl})
    MERGE (r)-[:HAS_PLANNING]->(p2)

    // --- Pagos ---
    WITH r, row
    UNWIND coalesce(row.payments, []) AS pay
    MERGE (p3:Payment {name: pay})
    MERGE (r)-[:ACCEPTS_PAYMENT]->(p3)

    // --- Opciones para niñxs ---
    WITH r, row
    UNWIND coalesce(row.children, []) AS ch
    MERGE (c3:ChildrenOption {name: ch})
    MERGE (r)-[:HAS_CHILDREN_OPTION]->(c3)

    // --- Reviews (0..n) como nodos +
    //     mantiene además r.user_review con la primera reseña
    WITH r, row
    UNWIND coalesce(row.user_reviews, []) AS txt
    WITH r, row, txt WHERE txt IS NOT NULL AND trim(txt) <> ''
    MERGE (rev:Review {place_id: row.place_id, text: txt})
      ON CREATE SET rev.createdAt = timestamp()
    MERGE (r)-[:HAS_REVIEW {source:'user'}]->(rev)
    """

    run(cypher, params={"rows": rows, "city_name": CITY_NAME, "city_aliases": CITY_ALIASES})

    # Centroide de la ciudad (lat/lon promedio de restaurantes)
    run("""
    MATCH (c:City {name: $city})
    OPTIONAL MATCH (r:Restaurant)-[:LOCATED_IN]->(c)
    WITH c, avg(r.latitude) AS lat, avg(r.longitude) AS lon
    SET c.latitude = lat,
        c.longitude = lon,
        c.location  = CASE
                        WHEN lat IS NOT NULL AND lon IS NOT NULL
                        THEN point({latitude: lat, longitude: lon})
                        ELSE c.location
                      END
    """, {"city": CITY_NAME})

    graph.refresh_schema()

# ---------------------------
# Helpers de consulta
# ---------------------------
def get_restaurants_by_city_name(name_or_alias: str):
    """
    Devuelve lista de mapas de propiedades de restaurantes para una ciudad por nombre o alias (case-insensitive).
    """
    res = run("""
        MATCH (c:City)
        WHERE toLower(c.name) = toLower($q)
           OR (c.aliases IS NOT NULL AND any(a IN c.aliases WHERE toLower(a) = toLower($q)))
        MATCH (r:Restaurant)-[:LOCATED_IN]->(c)
        RETURN r
        ORDER BY r.title
    """, {"q": name_or_alias})
    return [row["r"] for row in res]

# ---------------------------
# Carga
# ---------------------------
ingest_dataframe(df)
graph.refresh_schema()

# Ejemplo de consulta (por alias de ciudad):
results = get_restaurants_by_city_name("Qro.")
print(f"Restaurantes en Qro.: {len(results)}")
for r in results:
    print(r.get("title"), '--->' ,r.get("rating"), '--->', r.get("address"))

Restaurantes en Qro.: 20
Al Sur Cocina Diversa ---> 4.8 ---> Calle José María Pino Suárez 211B, Centro, 76000 Santiago de Querétaro, Qro., Mexico
Amor Del Bueno ---> 4.7 ---> Punto Inn, Senda del Amanecer 51-Local 02, Milenio III, 76070 Santiago de Querétaro, Qro., Mexico
Antojos Veganos by VegCo ---> 4.8 ---> Cerro Azul 101, Colinas del Cimatario, 76090 Santiago de Querétaro, Qro., Mexico
Azu's Garden ---> 4.6 ---> C. 5 de Mayo 80, Centro Histórico, Centro, 76000 Santiago de Querétaro, Qro., Mexico
Be Green ---> 4.5 ---> Fray Diego de Landa 202-Local C, Quintas del Marques, 76047 Santiago de Querétaro, Qro., Mexico
COCINA LA CAYENA ---> 4.9 ---> Av. Las Torres 103, Los Virreyes, 76175 Santiago de Querétaro, Qro., Mexico
Carbónico ---> 4.7 ---> Camino Real de Carretas 241, Milenio III, 76060 Santiago de Querétaro, Qro., Mexico
Clorofila vegetarian restaurant ---> 4.8 ---> Calle Ezequiel Montes 34, Centro, 76000 Santiago de Querétaro, Qro., Mexico
Don Chilaquil suc Epigmenio ---> 4.7 --

In [139]:
def create_near_edges_haversine(graph, meters=1000, mode="walk", city=None, topk=None, debug=False):
    """
    Crea/actualiza aristas :IS_WALKING entre restaurantes dentro de 'meters'
    usando Haversine en Cypher (sin 'distance' ni 'point.distance' ni 'pow'; Neo4j 5).

    Parámetros:
      - graph: instancia Neo4jGraph() conectada.
      - meters (float): radio en metros (p. ej. 1000).
      - mode (str): SE IGNORA para el tipo de arista (se mantiene por compatibilidad).
      - city (str|None): nombre o alias de ciudad ("Qro.", "Querétaro"); None = toda la BD.
      - topk (int|None): si se indica, conserva solo los k vecinos más cercanos por nodo.
      - debug (bool): imprime el Cypher que se ejecuta.

    Propiedades en la arista :IS_WALKING:
      - e.meters, e.km, e.time_min, e.createdAt/e.updatedAt
    """

    # 1) Asegura point() si hay lat/lon (útil para otras consultas)
    graph.query("""
    MATCH (r:Restaurant)
    WHERE r.location IS NULL AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL
    SET r.location = point({latitude: toFloat(r.latitude), longitude: toFloat(r.longitude)})
    """)

    # 2) Velocidad fija de caminata (m/min) para e.time_min
    speed_mpm = 80.0  # ~4.8 km/h

    # 3) Ámbito por ciudad (opcional)
    if city is not None:
        city_filter = """
        MATCH (c:City)
        WHERE toLower(c.name) = toLower($city)
           OR (c.aliases IS NOT NULL AND any(a IN c.aliases WHERE toLower(a) = toLower($city)))
        WITH c
        MATCH (c)<-[:LOCATED_IN]-(r1:Restaurant)
        WHERE r1.latitude IS NOT NULL AND r1.longitude IS NOT NULL
        """
        from_anchor = "r1"
        match_r2 = """
        MATCH (r2:Restaurant)-[:LOCATED_IN]->(c)
        WHERE r2.latitude IS NOT NULL AND r2.longitude IS NOT NULL
          AND id(r2) > id(r1)
        """
    else:
        city_filter = """
        MATCH (r1:Restaurant)
        WHERE r1.latitude IS NOT NULL AND r1.longitude IS NOT NULL
        """
        from_anchor = "r1"
        match_r2 = """
        MATCH (r2:Restaurant)
        WHERE r2.latitude IS NOT NULL AND r2.longitude IS NOT NULL
          AND id(r2) > id(r1)
        """

    # 4) Crear/actualizar aristas :IS_WALKING (una sola vez por par con id(r2) > id(r1))
    cypher_create = f"""
    {city_filter}
    WITH {from_anchor} AS r1, $meters AS meters, $speed AS speed
    WITH r1, meters, speed,
         (meters / 111320.0) AS degLat,
         (meters / (111320.0 * cos(radians(r1.latitude)))) AS degLon
    {match_r2}
      AND r2.latitude  >= r1.latitude  - degLat AND r2.latitude  <= r1.latitude  + degLat
      AND r2.longitude >= r1.longitude - degLon AND r2.longitude <= r1.longitude + degLon

    // Haversine sin 'pow'
    WITH r1, r2, meters, speed,
         radians(r2.latitude - r1.latitude)  AS dLat,
         radians(r2.longitude - r1.longitude) AS dLon,
         radians(r1.latitude)  AS r1lat,
         radians(r2.latitude)  AS r2lat
    WITH r1, r2, meters, speed, dLat, dLon, r1lat, r2lat,
         sin(dLat/2.0) AS sdlat, sin(dLon/2.0) AS sdlon
    WITH r1, r2, meters, speed,
         2 * 6371000.0 * asin( sqrt( sdlat*sdlat + cos(r1lat) * cos(r2lat) * sdlon*sdlon ) ) AS d
    WHERE d <= meters
    MERGE (r1)-[e:IS_WALKING]->(r2)
    ON CREATE SET
        e.meters    = d,
        e.km        = d/1000.0,
        e.time_min  = d/speed,
        e.createdAt = timestamp()
    ON MATCH SET
        e.meters    = d,
        e.km        = d/1000.0,
        e.time_min  = d/speed,
        e.updatedAt = timestamp()
    """

    if debug:
        print("=== Cypher (:IS_WALKING / Haversine) ===")
        print(cypher_create)

    graph.query(cypher_create, {
        "meters": float(meters),
        "city": city,
        "speed": float(speed_mpm),
    })

    # 5) (Opcional) Limita a top-k vecinos más cercanos por nodo (solo aristas salientes)
    if isinstance(topk, int) and topk > 0:
        graph.query("""
        MATCH (r:Restaurant)-[e:IS_WALKING]->(:Restaurant)
        WITH r, e
        ORDER BY r.place_id, e.meters ASC
        WITH r, collect(e) AS rels
        WITH r, rels[toInteger($k) .. ] AS toDrop
        FOREACH (e IN toDrop | DELETE e)
        """, {"k": int(topk)})

    # 6) Estadísticas
    stats = graph.query("""
    MATCH ()-[e:IS_WALKING]->()
    RETURN count(e) AS edges, round(avg(e.meters)) AS avg_m, min(e.meters) AS min_m, max(e.meters) AS max_m
    """)
    return stats[0] if stats else {"edges": 0, "avg_m": None, "min_m": None, "max_m": None}

In [140]:
summary = create_near_edges_haversine(graph, meters=1000, mode="walk")
print(summary) 

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function: `id`.} {position: line: 13, column: 15, offset: 419} for query: "\n    \n        MATCH (r1:Restaurant)\n        WHERE r1.latitude IS NOT NULL AND r1.longitude IS NOT NULL\n        \n    WITH r1 AS r1, $meters AS meters, $speed AS speed\n    WITH r1, meters, speed,\n         (meters / 111320.0) AS degLat,\n         (meters / (111320.0 * cos(radians(r1.latitude)))) AS degLon\n    \n        MATCH (r2:Restaurant)\n        WHERE r2.latitude IS NOT NULL AND r2.longitude IS NOT NULL\n          AND id(r2) > id(r1)\n        \n      AND r2.latitude  >= r1.latitude  - degLat AND r2.latitude  <= r1.latitude  + degLat\n      AND r2.longitude >= r1.longitude - degLon AND r2.longitude <= r1.longitude + degLon\n\n    // Haversi

{'edges': 32, 'avg_m': 588.0, 'min_m': 42.605949951589814, 'max_m': 990.3760632842547}


In [57]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [26]:
!pip install -U "langchain-openai>=0.2.0,<0.4"

  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 0.1.16
    Uninstalling langchain-openai-0.1.16:
      Successfully uninstalled langchain-openai-0.1.16

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

In [116]:
from langchain_core.prompts import PromptTemplate


In [134]:
GENERIC_CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""
Eres experto en Cypher para Neo4j 5.
Devuelve **solo** un bloque:
```cypher
<CONSULTA CYPHER>
```
Reglas:
	•	Para “caminando / a pie / cerca / distancia”, usar arista no dirigida :IS_WALKING. 
    • Si el usuario pide los restaurantes que están caminando de un restaurante, usa la siguietne query como base: 
     	 MATCH (v:Restaurant) WHERE toLower(v.title)=toLower($TITLE)
		MATCH (v)-[e:IS_WALKING]-(n:Restaurant)
		RETURN n.title AS name, round(e.meters) AS meters, round(e.time_min,1) AS minutes, n.address AS address
		ORDER BY meters ASC
	•	Ancla por título de restaurante:
		MATCH (v:Restaurant) WHERE toLower(v.title) = toLower($TITLE)
	•	Si mencionan “Querétaro/Qro/Queretarock”, hace referencia a "Santiago de Querétaro".
	•	Ordena y limita resultados razonablemente
Esquema:
{schema}

Pregunta:
{question}
"""
)


In [135]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    top_k=20,
    allow_dangerous_requests=True,
    cypher_prompt=GENERIC_CYPHER_PROMPT,

)

In [136]:
res = chain.invoke({"query": "Dime lo restaurantes que están caminando de Azu's Garden usa IS_WALKIG"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (v:Restaurant) WHERE toLower(v.title) = toLower("Azu's Garden")
MATCH (v)-[e:IS_WALKING]-(n:Restaurant)
RETURN n.title AS name, round(e.meters) AS meters, round(e.time_min,1) AS minutes, n.address AS address
ORDER BY meters ASC

Full Context:
[{'name': 'Tacogreen', 'meters': 219.0, 'minutes': 2.7, 'address': 'manuel gutierrez najera sur #22, La Cruz, 76020 Santiago de Querétaro, Qro., Mexico'}, {'name': 'La Biznaga Arte y Café', 'meters': 234.0, 'minutes': 2.9, 'address': 'Manuel Gutiérrez Nájera 17, La Santa Cruz, La Cruz, 76020 Santiago de Querétaro, Qro., Mexico'}, {'name': 'VegCo', 'meters': 786.0, 'minutes': 9.8, 'address': 'J. Asunción Romero 1, San Javier, 76020 Santiago de Querétaro, Qro., Mexico'}, {'name': 'Vanggie', 'meters': 792.0, 'minutes': 9.9, 'address': 'Hidalgo 25-Local 7, Centro Histórico, Centro, 76000 Santiago de Querétaro, Qro., Mexico'}, {'name': 'Mixe Cocina Evolutiva', 'meters': 981.0, 

In [112]:
# Crea la chain
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    top_k=20,
    allow_dangerous_requests=True

)


In [114]:
res = chain.invoke({"query": "Dime lo restaurantes que están caminando de Azu's Garden"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Restaurant)-[:LOCATED_IN]->(:City {name: "Azu's Garden"})
RETURN r
Full Context:
[]

> Finished chain.
Lo siento, no tengo esa información.


In [ ]:
res = chain.invoke({"query": "Devuélveme 3 restaurantes con buena calificación en Santiago de Querétaro. La respuesta debe incluir el nombre, dirección, calificación y número de reseñas de cada restaurante."})
print(res["result"])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City{name: "Santiago de Querétaro"})
WHERE r.rating >= 4.5
RETURN r.title AS name, r.address AS address, r.rating AS rating, r.reviews AS reviews
LIMIT 3;
Full Context:
[{'name': "Azu's Garden", 'address': 'C. 5 de Mayo 80, Centro Histórico, Centro, 76000 Santiago de Querétaro, Qro., Mexico', 'rating': 4.6, 'reviews': 1149}, {'name': 'Al Sur Cocina Diversa', 'address': 'Calle José María Pino Suárez 211B, Centro, 76000 Santiago de Querétaro, Qro., Mexico', 'rating': 4.8, 'reviews': 55}, {'name': 'COCINA LA CAYENA', 'address': 'Av. Las Torres 103, Los Virreyes, 76175 Santiago de Querétaro, Qro., Mexico', 'rating': 4.9, 'reviews': 21}]

> Finished chain.
Azu's Garden - C. 5 de Mayo 80, Centro Histórico, Centro, 76000 Santiago de Querétaro, Qro., Mexico - Rating: 4.6, Reviews: 1149
Al Sur Cocina Diversa - Calle José María Pino Suárez 211B, Centro, 76000 Santiago de Querétaro, Qro., Mexico 

In [52]:
res = chain.invoke({"query": "Devuelveme la latitude y longitude de VegCo?"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Restaurant {title: "VegCo"}) 
RETURN r.latitude, r.longitude;
Full Context:
[{'r.latitude': 20.597718999999998, 'r.longitude': -100.3809309}]

> Finished chain.
La latitud es 20.597719 y la longitud es -100.380931 de VegCo.


In [ ]:
res = chain.invoke({"query": "Dime los tipos de comida de los restaurantes?"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Restaurant)-[:OF_TYPE]->(c:Category)
RETURN DISTINCT c.name;
Full Context:
[{'c.name': 'Vegan restaurant'}, {'c.name': 'Restaurant'}, {'c.name': 'Vegetarian restaurant'}, {'c.name': 'Family restaurant'}, {'c.name': 'Brunch restaurant'}, {'c.name': 'Asian restaurant'}, {'c.name': 'Sushi takeaway'}, {'c.name': 'Health food restaurant'}, {'c.name': 'Bar & grill'}]

> Finished chain.
Restaurante vegano, Restaurante, Restaurante vegetariano, Restaurante familiar, Restaurante de brunch, Restaurante asiático, Sushi para llevar, Restaurante de comida saludable, Bar & grill.


In [93]:
res = chain.invoke({"query": "Dame negocios  en Santiago de Querétaro?"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City)
WHERE c.name = 'Santiago de Querétaro'
RETURN r.title, r.address, r.phone, r.website, r.rating, r.reviews, r.price
Full Context:
[{'r.title': "Azu's Garden", 'r.address': 'C. 5 de Mayo 80, Centro Histórico, Centro, 76000 Santiago de Querétaro, Qro., Mexico', 'r.phone': '+52 427 137 3133', 'r.website': None, 'r.rating': 4.6, 'r.reviews': 1149, 'r.price': 'MX$200–300'}, {'r.title': 'Al Sur Cocina Diversa', 'r.address': 'Calle José María Pino Suárez 211B, Centro, 76000 Santiago de Querétaro, Qro., Mexico', 'r.phone': '+52 442 876 7780', 'r.website': None, 'r.rating': 4.8, 'r.reviews': 55, 'r.price': 'MX$100–200'}, {'r.title': 'COCINA LA CAYENA', 'r.address': 'Av. Las Torres 103, Los Virreyes, 76175 Santiago de Querétaro, Qro., Mexico', 'r.phone': '+52 442 130 0266', 'r.website': 'https://www.facebook.com/profile.php?id=100083854195601&mibextid=LQQJ4d', 'r.rating': 4.9, 'r.reviews': 

In [100]:
res = chain.invoke({"query": "Dame los restaurantes que están caminando de VegCo"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (r:Restaurant)-[:LOCATED_IN]->(:City {name: 'VegCo'})
RETURN r
Full Context:
[]

> Finished chain.
No sé la respuesta.


In [107]:
from langchain_core.prompts import PromptTemplate
#from langchain_neo4j.chains.graph_qa.cypher import GraphCypherQAChain
CYPHER_PROMPT_WALK = PromptTemplate(
    # El chain le pasa 'schema' y 'question'; añadimos 'TITLE' para filtrar por nombre.
    input_variables=["schema", "question", "TITLE"],
    template="""
Eres experto en Cypher para Neo4j 5.

Instrucciones OBLIGATORIAS:
- Devuelve **únicamente** un bloque de código con formato:
```cypher
<CONSULTA CYPHER>
Reglas:
- Si el usuario dice "caminando", "a pie" o similar, usa relaciones :NEAR con {{mode:"walk"}}.
- Las aristas NEAR se crean en una sola dirección; por eso SIEMPRE usa patrón no dirigido:
  (a)-[e:NEAR {{mode:"walk"}}]-(b)
- Si menciona un restaurante, filtra por título de forma case-insensitive:
  MATCH (v:Restaurant) WHERE toLower(v.title) = toLower($TITLE)

Genera SOLO el Cypher (sin explicación). Si preguntan “restaurantes caminando desde X”,
usa algo como:
MATCH (v:Restaurant) WHERE toLower(v.title) = toLower($TITLE)
MATCH (v)-[e:NEAR {{mode:"walk"}}]-(n:Restaurant)
RETURN n.title AS name, round(e.meters) AS meters, round(e.time_min,1) AS minutes
ORDER BY meters ASC

Esquema:
{schema}

Pregunta:
{question}
"""
)


chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=CYPHER_PROMPT_WALK,
        verbose=True,
    validate_cypher=True,
    allow_dangerous_requests=True
   # top_k=20,
)

# Ejemplo de invocación
res = chain.invoke({"query": "Dame los restaurantes que están caminando de VegCo",
                    "TITLE": "VegCo"})
print(res["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:

Full Context:
[]

> Finished chain.
No sé la respuesta.
